# Evaluación de aprendizaje por refuerzo — Entrega 3

Responsable: Ángel Uriel Martínez Godínez.

Configuración de evaluación: **50 episodios**, semillas **1000–1049**. Conserva los nombres comunes de resultados en
esultados/entrega3/.


In [1]:
# Preparación para Google Colab: descarga la versión pública actual del repositorio.
from pathlib import Path
import os
import subprocess
import sys

repo = Path('/content/StyleBot')
if not repo.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/krysto9872/StyleBot.git', str(repo)])

os.chdir(repo)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium'])
print('Directorio actual:', Path.cwd())


Directorio actual: /content/StyleBot


In [2]:
"""Evaluación reproducible de la Entrega 3 de StyleBot.

Responsable: Ángel Uriel Martínez Godínez.

Este archivo cubre el agente aleatorio y deja listas las métricas, CSV y
gráficas. No inventa resultados de Q-Learning ni Dyna-Q: esos se integran
cuando Luis entregue sus tablas o archivos de métricas.

Ejecución desde la raíz del repositorio:
    python 03_evaluacion_rl.py
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.entrega3.entorno_almacen import AlmacenStyleBot


# ---------------------------------------------------------------------------
# Protocolo común: no modificar al comparar agentes.
# ---------------------------------------------------------------------------
# Evaluación común solicitada por el líder: 50 episodios, semillas 1000--1049.
SEMILLA = 1000
EPISODIOS_EVALUACION = 50
PRENDAS_POR_EPISODIO = 5
CAPACIDAD_ESTANTE = 8
PROB_ESTANTE_LLENO = 0.10
MAX_PASOS = 60 * PRENDAS_POR_EPISODIO

_CWD = Path.cwd().resolve()
RAIZ = _CWD if (_CWD / "src" / "entrega3" / "entorno_almacen.py").exists() else Path(__file__).resolve().parents[2]
RESULTADOS = RAIZ / "resultados" / "entrega3"
RESULTADOS.mkdir(parents=True, exist_ok=True)


def datos_evaluacion_perfectos(n_por_clase: int = 100) -> tuple[np.ndarray, np.ndarray]:
    """Crea prendas balanceadas con percepción perfecta.

    Sirve para medir el comportamiento de la política sin mezclar errores de
    la CNN. Cuando exista el archivo de predicciones de la CNN, reemplaza este
    par de arreglos por ``etiquetas_reales, predicciones_cnn``.
    """
    etiquetas = np.repeat(np.arange(10, dtype=np.int64), n_por_clase)
    return etiquetas, etiquetas.copy()


def politica_aleatoria(_estado: int, entorno: AlmacenStyleBot, rng: np.random.Generator) -> int:
    """Línea base: selecciona una de las seis acciones sin usar el estado."""
    return int(rng.integers(entorno.action_space.n))


def politica_desde_tabla_q(tabla_q: np.ndarray) -> Callable[[int, AlmacenStyleBot, np.random.Generator], int]:
    """Convierte una tabla Q ya entrenada en una política codiciosa.

    Luis puede entregar una matriz de forma (1408, 6) en formato .npy. Esta
    función se utilizará después para evaluar Q-Learning y Dyna-Q con las
    mismas semillas que el agente aleatorio.
    """
    tabla_q = np.asarray(tabla_q, dtype=float)
    if tabla_q.shape != (1408, 6):
        raise ValueError("La tabla Q debe tener forma (1408, 6).")

    def politica(estado: int, _entorno: AlmacenStyleBot, rng: np.random.Generator) -> int:
        mejores = np.flatnonzero(tabla_q[int(estado)] == tabla_q[int(estado)].max())
        return int(rng.choice(mejores))

    return politica


def ejecutar_episodio(
    entorno: AlmacenStyleBot,
    politica: Callable[[int, AlmacenStyleBot, np.random.Generator], int],
    semilla: int,
    nombre_agente: str,
) -> dict:
    """Ejecuta un episodio y devuelve las cuatro métricas y sus fallos."""
    estado, info = entorno.reset(seed=semilla)
    rng = np.random.default_rng(semilla)
    recompensa_total = 0.0
    terminado = truncado = False

    while not (terminado or truncado):
        accion = politica(estado, entorno, rng)
        estado, recompensa, terminado, truncado, info = entorno.step(accion)
        recompensa_total += recompensa

    total = int(entorno.total_prendas)
    correctas = int(info["correctas"])
    exito_prendas = correctas / total if total else 0.0
    exito_episodio = int(
        terminado
        and correctas == total
        and info["incorrectas"] == 0
        and info["devueltas"] == 0
    )

    return {
        "agente": nombre_agente,
        "semilla": semilla,
        "recompensa_total": recompensa_total,
        "exito_prendas": exito_prendas,
        "exito_episodio": exito_episodio,
        "pasos": int(info["pasos"]),
        "terminado": bool(terminado),
        "truncado": bool(truncado),
        "correctas": correctas,
        "incorrectas": int(info["incorrectas"]),
        "fallos_cnn": int(info["fallos_cnn"]),
        "fallos_politica": int(info["fallos_politica"]),
        "acciones_invalidas": int(info["acciones_invalidas"]),
        "intentos_estante_lleno": int(info["intentos_estante_lleno"]),
        "devueltas": int(info["devueltas"]),
    }


def evaluar_agente(
    nombre_agente: str,
    politica: Callable[[int, AlmacenStyleBot, np.random.Generator], int],
    etiquetas: np.ndarray,
    predicciones: np.ndarray,
    episodios: int = EPISODIOS_EVALUACION,
) -> pd.DataFrame:
    """Evalúa con semillas 1000..1049, iguales para todos los agentes."""
    filas = []
    for episodio in range(episodios):
        entorno = AlmacenStyleBot(
            etiquetas=etiquetas,
            predicciones=predicciones,
            prendas_por_episodio=PRENDAS_POR_EPISODIO,
            capacidad=CAPACIDAD_ESTANTE,
            prob_estante_lleno=PROB_ESTANTE_LLENO,
            max_pasos=MAX_PASOS,
        )
        filas.append(
            ejecutar_episodio(entorno, politica, SEMILLA + episodio, nombre_agente)
        )
    return pd.DataFrame(filas)


def resumir_metricas(episodios: pd.DataFrame) -> pd.DataFrame:
    """Resume recompensa, éxito, pasos y todos los tipos de error."""
    metricas = {
        "episodios": len(episodios),
        "recompensa_promedio": episodios["recompensa_total"].mean(),
        "recompensa_desv": episodios["recompensa_total"].std(ddof=0),
        "tasa_exito_prendas": episodios["exito_prendas"].mean(),
        "tasa_exito_episodios": episodios["exito_episodio"].mean(),
        "pasos_promedio": episodios["pasos"].mean(),
        "incorrectas_promedio": episodios["incorrectas"].mean(),
        "fallos_cnn_promedio": episodios["fallos_cnn"].mean(),
        "fallos_politica_promedio": episodios["fallos_politica"].mean(),
        "acciones_invalidas_promedio": episodios["acciones_invalidas"].mean(),
        "intentos_estante_lleno_promedio": episodios["intentos_estante_lleno"].mean(),
        "devueltas_promedio": episodios["devueltas"].mean(),
    }
    return pd.DataFrame([metricas])


def guardar_graficas(episodios: pd.DataFrame) -> None:
    """Guarda las gráficas con los nombres comunes de la entrega."""
    x = np.arange(1, len(episodios) + 1)
    ventana = min(10, len(episodios))

    fig, eje = plt.subplots(figsize=(9, 4.5))
    eje.plot(x, episodios["recompensa_total"], alpha=0.35, label="episodio")
    eje.plot(x, episodios["recompensa_total"].rolling(ventana, min_periods=1).mean(), linewidth=2, label=f"media móvil ({ventana})")
    eje.set(title="Curva de recompensa: agente aleatorio", xlabel="Episodio", ylabel="Recompensa total")
    eje.grid(alpha=0.25)
    eje.legend()
    fig.tight_layout()
    fig.savefig(RESULTADOS / "curva_recompensa.png", dpi=160)
    plt.close(fig)

    # Un solo archivo común contiene éxito y pasos promedio para no alterar rutas.
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
    ejes[0].plot(x, episodios["exito_prendas"], alpha=0.35, label="episodio")
    ejes[0].plot(x, episodios["exito_prendas"].rolling(ventana, min_periods=1).mean(), linewidth=2, label=f"media móvil ({ventana})")
    ejes[0].set(title="Tasa de éxito", xlabel="Episodio", ylabel="Proporción correcta", ylim=(-0.05, 1.05))
    ejes[0].grid(alpha=0.25)
    ejes[0].legend()

    ejes[1].plot(x, episodios["pasos"], alpha=0.35, label="episodio")
    ejes[1].plot(x, episodios["pasos"].rolling(ventana, min_periods=1).mean(), linewidth=2, label=f"media móvil ({ventana})")
    ejes[1].set(title="Pasos por episodio", xlabel="Episodio", ylabel="Pasos")
    ejes[1].grid(alpha=0.25)
    ejes[1].legend()

    fig.suptitle("Evaluación del agente aleatorio (semillas 1000–1049)")
    fig.tight_layout()
    fig.savefig(RESULTADOS / "comparacion_tasa_exito.png", dpi=160)
    plt.close(fig)


def guardar_plantillas(resultado_aleatorio: pd.DataFrame, resumen_aleatorio: pd.DataFrame) -> None:
    """Guarda el avance independiente y formatos para la entrega de Luis."""
    resultado_aleatorio.to_csv(RESULTADOS / "episodios_agente_aleatorio.csv", index=False)
    resumen_aleatorio.assign(agente="Aleatorio", estado="completo").to_csv(
        RESULTADOS / "comparacion_agentes.csv", index=False
    )

    columnas_hiperparametros = [
        "algoritmo", "configuracion", "alpha", "gamma", "epsilon_inicial",
        "epsilon_minimo", "decaimiento_epsilon", "pasos_planificacion",
        "episodios_entrenamiento", "recompensa_promedio", "tasa_exito_prendas",
        "pasos_promedio", "semilla", "responsable", "estado",
    ]
    pd.DataFrame(columns=columnas_hiperparametros).to_csv(
        RESULTADOS / "configuraciones_hiperparametros.csv", index=False
    )

    analisis = resultado_aleatorio[[
        "agente", "semilla", "incorrectas", "fallos_cnn", "fallos_politica",
        "acciones_invalidas", "intentos_estante_lleno", "devueltas",
    ]].copy()
    analisis["nota"] = (
        "Línea base con percepción perfecta: fallos_cnn debe ser 0. "
        "El análisis final requiere predicciones de la CNN."
    )
    analisis.to_csv(RESULTADOS / "analisis_fallos.csv", index=False)


def main() -> None:
    etiquetas, predicciones = datos_evaluacion_perfectos()
    episodios = evaluar_agente("Aleatorio", politica_aleatoria, etiquetas, predicciones)
    resumen = resumir_metricas(episodios)

    guardar_graficas(episodios)
    guardar_plantillas(episodios, resumen)

    protocolo = {
        "semilla": SEMILLA,
        "episodios_evaluacion": EPISODIOS_EVALUACION,
        "prendas_por_episodio": PRENDAS_POR_EPISODIO,
        "capacidad_estante": CAPACIDAD_ESTANTE,
        "probabilidad_estante_lleno": PROB_ESTANTE_LLENO,
        "max_pasos": MAX_PASOS,
        "percepcion": "perfecta para aislar la línea base aleatoria",
        "pendiente_de_luis": ["tablas Q de Q-Learning", "tablas Q de Dyna-Q", "resultados de hiperparámetros"],
    }
    (RESULTADOS / "protocolo_evaluacion_aleatorio.json").write_text(
        json.dumps(protocolo, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    print("\nLÍNEA BASE ALEATORIA COMPLETADA")
    print(resumen.round(4).to_string(index=False))
    print(f"\nResultados guardados en: {RESULTADOS}")
    print("Pendiente de Luis: resultados/tablas Q de Q-Learning y Dyna-Q.")


if __name__ == "__main__":
    main()



LÍNEA BASE ALEATORIA COMPLETADA
 episodios  recompensa_promedio  recompensa_desv  tasa_exito_prendas  tasa_exito_episodios  pasos_promedio  incorrectas_promedio  fallos_cnn_promedio  fallos_politica_promedio  acciones_invalidas_promedio  intentos_estante_lleno_promedio  devueltas_promedio
        50              -454.22          11.3548               0.016                   0.0           300.0                  0.54                  0.0                      0.54                       150.92                             0.16                0.04

Resultados guardados en: /content/StyleBot/resultados/entrega3
Pendiente de Luis: resultados/tablas Q de Q-Learning y Dyna-Q.
